# Sample Size Calculator for A/B Testing

이 노트북은 커머스/구독 서비스 A/B 테스트에서 자주 쓰는 샘플 사이즈 계산 예제입니다.

다루는 내용:
- Binary metric: 구매 전환율, CTR, 해지율
- Continuous metric: 주문금액, 매출, 세션 시간
- MDE 변화에 따른 필요 샘플 수 변화
- 과거 데이터 또는 synthetic data 기반 baseline/variance 추정

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from math import ceil
from scipy.stats import norm

pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

## 1. Sample Size Functions

### Binary Metric

예: 구매 전환율, CTR, 해지율

N = (Z_alpha/2 + Z_beta)^2 * [p1(1-p1) + p2(1-p2)] / (p2-p1)^2

In [ ]:
def sample_size_binary(p1: float, p2: float, alpha: float = 0.05, power: float = 0.8) -> int:
    """Return required sample size per group for a two-sided test of proportions."""
    if not (0 < p1 < 1 and 0 < p2 < 1):
        raise ValueError("p1 and p2 must be between 0 and 1.")
    if p1 == p2:
        raise ValueError("p1 and p2 must be different.")

    z_alpha = norm.ppf(1 - alpha / 2)
    z_beta = norm.ppf(power)
    numerator = (z_alpha + z_beta) ** 2 * (p1 * (1 - p1) + p2 * (1 - p2))
    denominator = (p2 - p1) ** 2
    return ceil(numerator / denominator)

### Continuous Metric

예: 주문금액, 사용자당 매출, 세션 시간

N = (Z_alpha/2 + Z_beta)^2 * 2 * sigma^2 / delta^2

In [ ]:
def sample_size_continuous(sigma: float, delta: float, alpha: float = 0.05, power: float = 0.8) -> int:
    """Return required sample size per group for a two-sided test of means."""
    if sigma <= 0 or delta <= 0:
        raise ValueError("sigma and delta must be positive.")

    z_alpha = norm.ppf(1 - alpha / 2)
    z_beta = norm.ppf(power)
    n = ((z_alpha + z_beta) ** 2 * 2 * sigma ** 2) / (delta ** 2)
    return ceil(n)

## 2. Example: Binary Metric

커머스 예시:
- Baseline purchase conversion rate = 5%
- 기대 uplift = +5% relative
- 즉, 5.00% → 5.25%

In [ ]:
baseline_cvr = 0.05
relative_uplift = 0.05
expected_cvr = baseline_cvr * (1 + relative_uplift)

n_per_group = sample_size_binary(baseline_cvr, expected_cvr)

print(f"Baseline CVR: {baseline_cvr:.2%}")
print(f"Expected CVR: {expected_cvr:.2%}")
print(f"Required sample size per group: {n_per_group:,}")
print(f"Total sample size: {n_per_group * 2:,}")

## 3. MDE Sensitivity

MDE가 작아질수록 필요한 샘플 수가 제곱으로 증가합니다.

In [ ]:
mde_relative_list = np.array([0.02, 0.03, 0.05, 0.07, 0.10, 0.15, 0.20])
rows = []

for rel_mde in mde_relative_list:
    p2 = baseline_cvr * (1 + rel_mde)
    n = sample_size_binary(baseline_cvr, p2)
    rows.append({
        "relative_mde": rel_mde,
        "control_rate": baseline_cvr,
        "treatment_rate": p2,
        "absolute_diff": p2 - baseline_cvr,
        "n_per_group": n,
        "total_n": n * 2
    })

mde_df = pd.DataFrame(rows)
mde_df

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(mde_df["relative_mde"] * 100, mde_df["n_per_group"], marker="o")
plt.xlabel("Relative MDE (%)")
plt.ylabel("Required Sample Size per Group")
plt.title("Sample Size vs Relative MDE")
plt.grid(True)
plt.show()

## 4. Example: Continuous Metric

커머스 예시:
- 사용자당 주문금액 평균 = 30,000원
- 표준편차 = 80,000원
- 탐지하고 싶은 최소 효과 = +1,000원

In [ ]:
sigma = 80_000
delta = 1_000

n_per_group_cont = sample_size_continuous(sigma=sigma, delta=delta)

print(f"Sigma: {sigma:,.0f}")
print(f"Delta: {delta:,.0f}")
print(f"Required sample size per group: {n_per_group_cont:,}")
print(f"Total sample size: {n_per_group_cont * 2:,}")

## 5. Load Commerce Dataset Optional

Kaggle 또는 공개 커머스 데이터를 사용할 경우, 아래처럼 파일 경로만 바꾸면 됩니다.

예시 컬럼 가정:
- user_id
- converted
- revenue

데이터셋이 없다면 synthetic data를 생성해서 실습합니다.

In [ ]:
DATA_PATH = "../data/ecommerce.csv"

if os.path.exists(DATA_PATH):
    df = pd.read_csv(DATA_PATH)
    print("Loaded:", DATA_PATH)
else:
    rng = np.random.default_rng(42)
    n_users = 100_000
    df = pd.DataFrame({
        "user_id": np.arange(1, n_users + 1),
        "visited": 1,
        "converted": rng.binomial(1, 0.05, n_users),
    })
    df["revenue"] = np.where(
        df["converted"] == 1,
        rng.lognormal(mean=np.log(30_000), sigma=1.0, size=n_users),
        0
    )
    print("Synthetic ecommerce data generated.")

df.head()

## 6. Estimate Baseline from Data

In [ ]:
baseline_conversion = df["converted"].mean()
revenue_sigma = df["revenue"].std()

print(f"Baseline conversion rate: {baseline_conversion:.2%}")
print(f"Revenue standard deviation: {revenue_sigma:,.0f}")

In [ ]:
relative_mde = 0.05
p1 = baseline_conversion
p2 = p1 * (1 + relative_mde)

n_binary = sample_size_binary(p1, p2)

print("[Binary Metric: Conversion Rate]")
print(f"p1: {p1:.2%}")
print(f"p2: {p2:.2%}")
print(f"n per group: {n_binary:,}")
print(f"total n: {n_binary * 2:,}")

delta = 1_000
n_cont = sample_size_continuous(revenue_sigma, delta)

print("\n[Continuous Metric: Revenue per User]")
print(f"sigma: {revenue_sigma:,.0f}")
print(f"delta: {delta:,.0f}")
print(f"n per group: {n_cont:,}")
print(f"total n: {n_cont * 2:,}")

## 7. Convert Sample Size to Experiment Duration

필요 샘플 수를 일별 트래픽으로 나누어 실험 기간을 추정합니다.

In [ ]:
daily_eligible_users = 20_000

required_days = ceil((n_binary * 2) / daily_eligible_users)

print(f"Daily eligible users: {daily_eligible_users:,}")
print(f"Required total users: {n_binary * 2:,}")
print(f"Estimated experiment duration: {required_days} days")

## 8. Practical Notes

- MDE는 통계가 아니라 비즈니스 판단에서 출발해야 한다.
- MDE가 작을수록 필요한 샘플 수는 제곱으로 증가한다.
- 분산이 큰 지표는 샘플 수가 크게 필요하다.
- 50:50 allocation이 가장 효율적이다.
- Power를 0.8에서 0.9로 높이면 샘플 수가 증가한다.
- 실험 전 AA test 또는 과거 데이터로 baseline/variance를 추정하는 것이 좋다.